### Imports

In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor

### Import Cleaned CSV

In [9]:
final_df = pd.read_csv('./new_dataset/cleaned_data_unscaled.csv')

y = final_df["productivity_score"]
X = final_df.drop(columns = ["productivity_score"])

### Split Data (Train and Test)

In [10]:
# split dataset into training and test sets (random_state ensures reproducible results)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42
)

print("Training set:", X_trainval.shape)
print("Test set:", X_test.shape)

Training set: (3600, 22)
Test set: (900, 22)


### K-Folds and Training

In [ ]:
cols_to_exclude = ["job_role_Designer", "job_role_Developer", "job_role_Manager", "job_role_Marketer", "job_role_Writer", "deadline_pressure_level_Low", "deadline_pressure_level_Medium", "deadline_pressure_level_High"]
cols_to_scale = X_trainval.columns.difference(cols_to_exclude)

# models go HERE!!!!
models = {
    "linear_regression": LinearRegression(),
    "ridge_l2": Ridge(alpha = 1.0),
    "decision_tree": DecisionTreeRegressor(random_state = 42),
    "xgboost": XGBRegressor(
        n_estimators = 500,
        learning_rate = 0.05,
        max_depth = 6,
        random_state = 42
    ),
    "neural_net": MLPRegressor(
        hidden_layer_sizes = (64, 32),
        activation = "relu",
        solver = "adam",
        alpha = 1e-4,
        learning_rate_init = 1e-3,
        max_iter = 500,
        early_stopping = True,
        random_state = 42
    )
}

k = 5
kf = KFold(n_splits = k, shuffle = True, random_state = 42)

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2"
}

preprocess = ColumnTransformer(
    transformers = [("num", StandardScaler(), cols_to_scale)],
    remainder = "passthrough"
)

for name, model in models.items():

    pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", model)
    ])

    scores = cross_validate(
        pipe,
        X_trainval,
        y_trainval,
        cv = kf,
        scoring = scoring
    )

    mae = -scores["test_mae"]
    rmse = -scores["test_rmse"]
    r2 = scores["test_r2"]

    print(f"\n{name}")
    print(f"MAE : {mae.mean():.4f} ± {mae.std():.4f}")
    print(f"RMSE: {rmse.mean():.4f} ± {rmse.std():.4f}")
    print(f"R^2 : {r2.mean():.4f} ± {r2.std():.4f}")


linear_regression
MAE : 4.0153 ± 0.0769
RMSE: 5.0267 ± 0.1508
R^2 : 0.8769 ± 0.0065

ridge_l2
MAE : 4.0150 ± 0.0763
RMSE: 5.0265 ± 0.1504
R^2 : 0.8769 ± 0.0065

decision_tree
MAE : 6.9210 ± 0.2776
RMSE: 8.7169 ± 0.3422
R^2 : 0.6301 ± 0.0186

xgboost
MAE : 4.4434 ± 0.1078
RMSE: 5.5968 ± 0.1512
R^2 : 0.8474 ± 0.0076

neural_net
MAE : 4.1338 ± 0.1345
RMSE: 5.2145 ± 0.2176
R^2 : 0.8675 ± 0.0097


### Final Evaluation

In [12]:
best_model_name = "ridge_l2"
best_model = models[best_model_name]

final_pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", best_model)
])

final_pipe.fit(X_trainval, y_trainval)
test_preds = final_pipe.predict(X_test)

mae = mean_absolute_error(y_test, test_preds)
rmse = np.sqrt(mean_squared_error(y_test, test_preds))
r2 = r2_score(y_test, test_preds)

print(f"Final Test on: {best_model_name}")
print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R^2 : {r2:.4f}")

ridge_model = final_pipe.named_steps["model"]
feature_names = final_pipe.named_steps["preprocess"].get_feature_names_out()

coefficients = ridge_model.coef_

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "abs_coefficient": np.abs(coefficients)
}).sort_values("abs_coefficient", ascending = False)

print("\nRidge coefficients:")
print(coef_df[["feature", "coefficient"]].to_string(index = False))

Final Test on: ridge_l2
MAE : 3.9424
RMSE: 4.9099
R^2 : 0.8788

Ridge coefficients:
                                  feature  coefficient
             num__tasks_automated_percent     7.432986
                    num__experience_years     5.691528
                 num__focus_hours_per_day     5.499874
                  num__burnout_risk_score    -4.644619
                  num__error_rate_percent    -4.637077
             num__work_life_balance_score    -0.982815
          num__manual_work_hours_per_week    -0.756252
   remainder__deadline_pressure_level_Low     0.675671
  remainder__deadline_pressure_level_High    -0.614288
        num__ai_tool_usage_hours_per_week    -0.599199
              num__meeting_hours_per_week    -0.457839
               remainder__job_role_Writer    -0.283908
             remainder__job_role_Marketer     0.173814
            remainder__job_role_Developer    -0.156782
             remainder__job_role_Designer     0.144786
              remainder__job_role_Ma

### Baseline (Mean Predictor)

In [13]:
# Baseline predicting the training mean for the test set
y_pred_baseline = np.full(len(y_test), y_trainval.mean())

baseline_mae = mean_absolute_error(y_test, y_pred_baseline)
baseline_rmse = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
baseline_r2 = r2_score(y_test, y_pred_baseline)

print(f"Baseline MAE: {baseline_mae:.2f}")
print(f"Baseline RMSE: {baseline_rmse:.2f}")
print(f"Baseline R^2: {baseline_r2:.4f}")

Baseline MAE: 11.25
Baseline RMSE: 14.11
Baseline R^2: -0.0004
